# Numerical Computation for Deep Learning — PyTorch Edition



## Learning objectives

By the end of this notebook, you should be able to:

- distinguish **precision** from the **dynamic range** of floating-point types;
- recognize overflow, underflow, loss of information, and catastrophic cancellation;
- understand why mathematically equivalent formulas may behave differently on a computer;
- use numerically stable PyTorch primitives (`logsumexp`, `log_softmax`, `cross_entropy`, `BCEWithLogitsLoss`, ...);
- diagnose `NaN`, `inf`, and problematic gradients;
- understand why FP16/BF16 and mixed precision require specific numerical care.

> **Guiding idea:** in deep learning, it is not enough for a formula to be mathematically correct. It must also be implemented stably with a finite number of bits.

> **Teaching format.** Each experiment is followed by an interpretation block: **Problem → Why it happens → Observed behavior/failure → Adopted solution → Takeaway**.


## Slides → PyTorch map

| Topic | Naive version / risk | Recommended PyTorch approach |
|---|---|---|
| `log(sum(exp(x)))` | overflow / underflow | `torch.logsumexp` |
| `log(softmax(x))` | saturation and `log(0)` | `F.log_softmax` |
| multiclass cross-entropy | softmax → log → mean | `F.cross_entropy` |
| binary cross-entropy | sigmoid → BCE | `F.binary_cross_entropy_with_logits` / `nn.BCEWithLogitsLoss` |
| normalization | division by an almost-zero standard deviation | `sqrt(var + eps)` in normalization layers |
| mixed precision | FP16 gradient underflow | `torch.autocast` + `torch.amp.GradScaler` |

The main sequence follows the part of the slides devoted to: numerical precision → rounding → overflow/underflow → subtraction → `log`/`sqrt` → log-sum-exp → softmax → cross-entropy → bug hunting.

## 0. Setup

In [ ]:
import math
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

torch.manual_seed(0)

## 1. Floating point: precision and dynamic range

A real number is represented using a finite number of bits. Two properties are especially important:

- **precision**: how finely we can distinguish nearby numbers;
- **dynamic range**: how small or large representable numbers can be.

`torch.finfo` lets us inspect the numerical properties of a dtype directly.

In [ ]:
dtypes = [torch.float16, torch.bfloat16, torch.float32, torch.float64]

for dtype in dtypes:
    f = torch.finfo(dtype)
    print(
        f"{str(dtype):15s} "
        f"bits={f.bits:2d}  "
        f"eps={f.eps:.3e}  "
        f"tiny={f.tiny:.3e}  "
        f"max={f.max:.3e}"
    )

#### Experiment interpretation

**Problem.** A floating-point dtype has both a finite **precision** and a finite **dynamic range**. Treating all 16-bit or 32-bit formats as interchangeable hides important failure modes.

**Why it happens.** `eps` measures the spacing of representable numbers around 1, while `tiny` and `max` describe the useful magnitude range. FP16 has a much smaller dynamic range than BF16/FP32, while BF16 trades mantissa precision for a large exponent range.

**Observed behavior.** The table shows that FP16 overflows at much smaller magnitudes, whereas BF16 has approximately the FP32 exponent range but coarser precision.

**Adopted solution.** Inspect `torch.finfo(dtype)` whenever the numerical scale of a computation matters, and choose the dtype according to both precision and range requirements.

**Takeaway.** *Bit width alone does not tell you how numerically safe a dtype is.*


### Observation

`eps` is the distance between 1 and the next representable number greater than 1. Therefore, there is no uniform “absolute precision” over the entire real line: the spacing between representable numbers grows with their magnitude.

**Question:** why does BF16 have a worse `eps` than FP16, while having a dramatically larger `max`?

## 2. Rounding: adding a small number may have no effect

This directly reproduces the example from the slides: in `float32`, `1 + 1e-8` may be rounded exactly to `1`.

In [ ]:
a = torch.tensor([0.0, 1e-8], dtype=torch.float32)

print("a              =", a)
print("a + 1          =", a + 1.0)
print("(a + 1) - 1   =", (a + 1.0) - 1.0)

#### Experiment interpretation

**Problem.** Adding a very small quantity to a much larger one can make the small quantity disappear completely.

**Why it happens.** Around 1.0, adjacent FP32 numbers are separated by about `1.19e-7`. The increment `1e-8` is smaller than that spacing, so `1 + 1e-8` rounds back to exactly `1`.

**Observed failure.** `(a + 1) - 1` returns zero for both entries. The original `1e-8` information has not merely been hidden; it has been lost during rounding.

**Adopted solution.** Avoid arithmetic that repeatedly mixes quantities with very different scales. When such small increments are meaningful, use a higher-precision accumulator or reformulate the computation.

**Takeaway.** *A mathematically reversible sequence of operations is not necessarily reversible in floating point.*


The last line is the key one: after rounding, the information carried by the small term is **not recoverable**.

In [ ]:
for dtype in [torch.float16, torch.bfloat16, torch.float32, torch.float64]:
    x = torch.tensor(1e-8, dtype=dtype)
    one = torch.tensor(1.0, dtype=dtype)
    print(dtype, "x =", x.item(), "  1+x =", (one + x).item())

#### Experiment interpretation

**Problem.** The same small value behaves differently across dtypes.

**Why it happens.** FP16 cannot even represent `1e-8` as a non-zero normal/subnormal value here, while BF16 and FP32 can represent a value near `1e-8` but still cannot preserve it when added to 1 because their spacing around 1 is too coarse. FP64 has enough precision for the increment to survive.

**Observed failure.** In FP16 the value itself becomes zero; in BF16/FP32 it exists in isolation but disappears in `1+x`.

**Adopted solution.** Distinguish **underflow of the value itself** from **loss of significance during an operation**. Increasing precision can help, but changing the numerical formulation is often preferable.

**Takeaway.** *Representability of `x` does not imply representability of `1+x`.*


## 3. Floating-point arithmetic is not associative

Over the real numbers:

$$
(a+b)+c = a+(b+c)
$$

In floating point, this is not necessarily true.

In [ ]:
a = torch.tensor(1e8, dtype=torch.float32)
b = torch.tensor(-1e8, dtype=torch.float32)
c = torch.tensor(1.0, dtype=torch.float32)

left = (a + b) + c
right = a + (b + c)

print("(a+b)+c =", left.item())
print("a+(b+c) =", right.item())

#### Experiment interpretation

**Problem.** Floating-point addition is not associative, even though real-number addition is.

**Why it happens.** In `b+c`, the small `1` is rounded away when it is combined with `-1e8`. Therefore `a+(b+c)` becomes zero. In `(a+b)+c`, the two large terms cancel exactly first, leaving the `1` intact.

**Observed failure.** Two algebraically equivalent parenthesizations produce `1.0` and `0.0`.

**Adopted solution.** Do not assume that changing reduction order is numerically neutral. Prefer stable summation strategies, pairwise/tree reductions, or higher-precision accumulation when small terms matter.

**Takeaway.** *Reordering operations can change floating-point results.*


This explains why reductions, parallel execution, and different backends can produce small differences even with identical inputs.

### Mini-experiment: summation order

In [ ]:
x = torch.cat([
    torch.tensor([1e8], dtype=torch.float32),
    torch.ones(100_000, dtype=torch.float32),
    torch.tensor([-1e8], dtype=torch.float32),
])

print("torch.sum float32:", x.sum().item())
print("sum after float64 cast:", x.double().sum().item())

#### Experiment interpretation

**Problem.** Summing many values of different scales can accumulate rounding error.

**Why it happens.** The large positive and negative values dominate the representable spacing. Depending on the reduction order, some of the unit-sized contributions are rounded away before the large terms cancel.

**Observed failure.** The FP32 reduction returns `99992` instead of the mathematically expected `100000`. Casting to FP64 before the reduction recovers the expected result in this example.

**Adopted solution.** Use numerically safer accumulation: higher-precision reduction, pairwise summation, compensated summation where appropriate, or rescale the data.

**Takeaway.** *The dtype of the accumulator matters as much as the dtype of the stored data.*


## 4. Overflow and underflow of `exp`

The slides point out that `exp(x)` overflows at around 89 in float32. Instead of memorizing that number, we can derive the threshold from the dtype:

$$
\exp(x) \leq \text{finfo.max} \quad\Rightarrow\quad x \lesssim \log(\text{finfo.max})
$$

In [ ]:
for dtype in [torch.float16, torch.bfloat16, torch.float32, torch.float64]:
    f = torch.finfo(dtype)
    print(dtype, "log(max) ≈", math.log(f.max))

#### Experiment interpretation

**Problem.** `exp(x)` has a dtype-dependent overflow threshold.

**Why it happens.** Overflow begins when `exp(x)` exceeds the largest finite representable number. Therefore the approximate threshold is `log(torch.finfo(dtype).max)`.

**Observed behavior.** FP16 can only tolerate inputs around 11 before overflow, while FP32/BF16 tolerate roughly 89 and FP64 roughly 710.

**Adopted solution.** Never rely on raw exponentials of unrestricted values. Shift inputs, work in the log domain, or use stabilized library functions such as `logsumexp`, `softmax`, and fused loss functions.

**Takeaway.** *The safe input range of nonlinear functions depends strongly on dtype.*


In [ ]:
values = torch.tensor([10.0, 12.0, 80.0, 89.0, 100.0])

for dtype in [torch.float16, torch.float32, torch.float64]:
    print("\n", dtype)
    print(torch.exp(values.to(dtype)))

#### Experiment interpretation

**Problem.** Direct exponentiation of moderately large logits can overflow to `inf`.

**Why it happens.** Exponential growth is extremely fast. In FP16, even `exp(12)` is already too large; in FP32, values around 89 overflow.

**Observed failure.** The experiment produces `inf` much earlier in FP16 than in FP32 or FP64.

**Adopted solution.** Avoid computing exponentials before normalization. For softmax-like expressions, subtract the maximum logit first or use PyTorch's stabilized implementations.

**Takeaway.** *A value that looks numerically modest as a logit can be catastrophic after `exp`.*


### Underflow

For very negative inputs, `exp(x)` becomes so small that it is rounded to zero.

In [ ]:
values = torch.tensor([-10.0, -50.0, -100.0, -1000.0])

for dtype in [torch.float16, torch.float32, torch.float64]:
    print("\n", dtype)
    print(torch.exp(values.to(dtype)))

#### Experiment interpretation

**Problem.** Very negative exponentials can underflow to zero.

**Why it happens.** `exp(x)` approaches zero rapidly for negative `x`. Once the result becomes smaller than what the dtype can represent, it is rounded to zero.

**Observed failure.** FP16 reaches zero already for values such as `-50`, while FP32 survives longer. If the result is later used inside `log`, as a denominator, or as a probability, the zero can trigger `-inf`, division by zero, or lost gradients.

**Adopted solution.** Keep calculations in the log domain whenever possible and use stable compound operators rather than materializing tiny probabilities explicitly.

**Takeaway.** *Underflow may look harmless until a later operation amplifies its consequences.*


## 5. Secondary effects: from `inf` to `NaN`

An overflow does not necessarily remain confined to the operation that produced it.

In [ ]:
x = torch.exp(torch.tensor(100.0, dtype=torch.float32))
y = torch.exp(torch.tensor(100.0, dtype=torch.float32))
z = x - y

print("x =", x)
print("y =", y)
print("x-y =", z)
print("isfinite?", torch.isfinite(z).item())

#### Experiment interpretation

**Problem.** A local overflow can propagate into a `NaN` elsewhere in the graph.

**Why it happens.** Both exponentials become `inf`; IEEE floating-point arithmetic defines `inf - inf` as undefined, producing `NaN`.

**Observed failure.** The first problematic operation is `exp`, but the visible failure appears later in the subtraction.

**Adopted solution.** Trace non-finite values back to their first occurrence with `torch.isfinite`, hooks, or anomaly detection. Stabilize the earliest unstable operation instead of patching the final `NaN`.

**Takeaway.** *The operation that produces `NaN` is not always the operation that caused the numerical instability.*


`inf - inf` has no well-defined value and produces `NaN`. This is a common pattern: the operation that produces the `NaN` may occur **much later** than the operation that caused the original overflow.

## 6. Catastrophic cancellation: the variance example

A mathematically valid formula for the variance is

$$
\operatorname{Var}(X) = E[X^2] - E[X]^2.
$$

If the two terms are large and nearly equal, the subtraction can cancel many significant digits.

In [ ]:
x = 1000.0 + 0.1 * torch.arange(10, dtype=torch.float32)

var_naive = (x * x).mean() - x.mean() ** 2
var_torch = torch.var(x, correction=0)
var_centered = ((x - x.mean()) ** 2).mean()

print("naive E[x²]-E[x]² =", var_naive)
print("centered form       =", var_centered)
print("torch.var           =", var_torch)

#### Experiment interpretation

**Problem.** The textbook identity `E[x²] - E[x]²` can be numerically disastrous for variance.

**Why it happens.** Both terms are around one million, while the true variance is only about `0.08`. Subtracting two nearly equal large numbers causes **catastrophic cancellation**: the leading digits cancel and the remaining low-order digits are dominated by rounding error.

**Observed failure.** The naive formula returns the impossible negative variance `-0.0625`, whereas the centered computation and `torch.var` return approximately `0.0825`.

**Adopted solution.** Compute variance with a stable centered/two-pass or online algorithm, or simply use `torch.var` rather than expanding the algebraic identity.

**Takeaway.** *Algebraic equivalence does not imply numerical equivalence.*


In [ ]:
print("sqrt(naive variance) =", torch.sqrt(var_naive))
print("sqrt(torch variance) =", torch.sqrt(var_torch))

#### Experiment interpretation

**Problem.** A small upstream error can become a hard domain error downstream.

**Why it happens.** Variance should be non-negative, but catastrophic cancellation produced a negative value. `sqrt` is undefined for negative real inputs, so PyTorch returns `NaN`.

**Observed failure.** `sqrt(var_naive)` is `NaN`, while `sqrt(torch.var(...))` is finite.

**Adopted solution.** Fix the variance calculation itself. Clamping a variance to zero can sometimes be a defensive last step, but it should not replace a stable variance algorithm.

**Takeaway.** *Domain violations are often symptoms of an earlier precision problem.*


This example connects three issues from the slides:

1. subtraction between quantities of similar magnitude;
2. a variance that can become numerically incorrect;
3. `sqrt` of a negative number → `NaN`.

### Comparison with float64

In [ ]:
x64 = x.double()
var_naive64 = (x64 * x64).mean() - x64.mean() ** 2
print("naive float64 =", var_naive64)

#### Experiment interpretation

**Problem.** Can more precision rescue an unstable formula?

**Why it happens.** FP64 has enough mantissa precision here to preserve the small difference between the two large variance terms.

**Observed behavior.** The naive variance formula becomes accurate after converting to FP64.

**Adopted solution.** Higher precision is a useful diagnostic and sometimes a valid engineering choice, but the preferred fix is still a stable algorithm because larger datasets or worse scaling can break the same formula again.

**Takeaway.** *Increasing precision can postpone cancellation; reformulating the computation removes the root cause.*


## 7. `log` and `sqrt`: domains and problematic derivatives

- `log(0) = -inf`;
- `log(x<0)` produces `NaN` over the reals;
- `sqrt(0)=0`, but the derivative $1/(2\sqrt{x})$ diverges at zero.

In [ ]:
vals = torch.tensor([1.0, 0.0, -1.0], dtype=torch.float32)
print("log :", torch.log(vals))
print("sqrt:", torch.sqrt(vals))

#### Experiment interpretation

**Problem.** `log` and `sqrt` have restricted real-valued domains.

**Why it happens.** `log(0)` tends to `-inf`, while `log(x<0)` and `sqrt(x<0)` are not real-valued. PyTorch therefore produces `-inf` or `NaN` for these inputs.

**Observed failure.** The tensor contains both `-inf` and `NaN`, depending on the operation and input.

**Adopted solution.** Ensure the mathematical preconditions of the operation hold. Use stable parameterizations, positive constraints, or carefully justified epsilons instead of blindly calling these functions on unconstrained values.

**Takeaway.** *Numerical stability starts with respecting the function's domain.*


### Autograd makes the derivative problem of `sqrt` visible

In [ ]:
x = torch.tensor(0.0, requires_grad=True)
y = torch.sqrt(x)
y.backward()

print("sqrt(0) =", y.item())
print("gradient =", x.grad)

#### Experiment interpretation

**Problem.** A finite forward value can still have a non-finite gradient.

**Why it happens.** The derivative of `sqrt(x)` is `1/(2*sqrt(x))`, which diverges as `x -> 0+`.

**Observed failure.** `sqrt(0)` is exactly zero, but autograd returns an infinite gradient.

**Adopted solution.** If zero is a legitimate input in a differentiable path, regularize the expression according to the model semantics, for example with `sqrt(x + eps)`, or use a formulation whose derivative remains controlled.

**Takeaway.** *Checking only forward activations is insufficient; gradients need numerical checks too.*


In [ ]:
x = torch.tensor(0.0, requires_grad=True)
eps = 1e-6
y = torch.sqrt(x + eps)
y.backward()

print("sqrt(eps) =", y.item())
print("gradient   =", x.grad.item())

#### Experiment interpretation

**Problem.** We want to prevent the infinite derivative at zero without introducing a large arbitrary distortion.

**Why it happens.** Adding a positive `eps` moves the evaluation point away from the singularity. The derivative becomes finite, although it can still be large.

**Observed behavior.** With `eps=1e-6`, the output is about `1e-3` and the gradient is about `500` instead of `inf`.

**Adopted solution.** Add a small positive stabilizer **when it is consistent with the intended formula**, and choose its scale with awareness of the dtype and expected data range.

**Takeaway.** *Epsilon regularization removes the singularity, but the epsilon value is part of the numerical model.*


## 8. Where should `eps` go?

The slides ask whether we should use

```python
sqrt(variance + eps)
```

or

```python
sqrt(variance) + eps
```

The two expressions are **not equivalent**. Normalization layers typically use the first form, with `eps` inside the square root.

In [ ]:
variance = torch.tensor([0.0, 1e-12, 1e-8, 1e-4, 1.0])
eps = 1e-5

inside = torch.sqrt(variance + eps)
outside = torch.sqrt(variance) + eps

print(torch.stack([variance, inside, outside], dim=1))

#### Experiment interpretation

**Problem.** `sqrt(var + eps)` and `sqrt(var) + eps` are often treated as interchangeable hacks, but they are different functions.

**Why it happens.** Putting `eps` inside the square root imposes a floor on the **variance** before converting to a standard deviation; adding it outside imposes a floor directly on the **standard deviation**. Their effect differs dramatically when variance is near zero.

**Observed behavior.** At zero variance the two denominators differ by more than two orders of magnitude for `eps=1e-5`.

**Adopted solution.** Use the formulation defined by the algorithm or layer. For normalization layers, the conventional stable form is based on `sqrt(variance + eps)`.

**Takeaway.** *Where epsilon is inserted changes both numerical stability and model behavior.*


When the variance is small, the position of `eps` changes the denominator substantially and therefore changes the function being implemented.

## 9. Specialized functions: `log1p` and `expm1`

For small $x$, computing `1+x` first may completely lose the increment. This is why specialized primitives exist:

- `torch.log1p(x)` computes $\log(1+x)$;
- `torch.expm1(x)` computes $e^x-1$.

In [ ]:
x = torch.tensor(1e-8, dtype=torch.float32)

print("torch.log(1+x) =", torch.log(1 + x).item())
print("torch.log1p(x)  =", torch.log1p(x).item())
print()
print("torch.exp(x)-1  =", (torch.exp(x) - 1).item())
print("torch.expm1(x)  =", torch.expm1(x).item())

#### Experiment interpretation

**Problem.** Naively evaluating `log(1+x)` or `exp(x)-1` loses tiny increments.

**Why it happens.** For small `x`, `1+x` may round to exactly 1 and `exp(x)` may round to exactly 1 before the subtraction/logarithm is applied.

**Observed failure.** The naive expressions return zero for `x=1e-8`, while `torch.log1p` and `torch.expm1` preserve a result close to `1e-8`.

**Adopted solution.** Use specialized functions such as `torch.log1p` and `torch.expm1`, which are implemented to maintain accuracy near zero.

**Takeaway.** *Library functions often encode numerically superior formulas for common cancellation-prone patterns.*


### Visualizing the error in `log(1+x)`

In [ ]:
xs = torch.logspace(-12, -2, 200, dtype=torch.float32)
naive = torch.log(1 + xs)
stable = torch.log1p(xs)
reference = torch.log1p(xs.double()).float()

err_naive = (naive - reference).abs()
err_stable = (stable - reference).abs()

plt.figure(figsize=(7, 4))
plt.loglog(xs.numpy(), (err_naive + 1e-30).numpy(), label="log(1+x)")
plt.loglog(xs.numpy(), (err_stable + 1e-30).numpy(), label="log1p(x)")
plt.xlabel("x")
plt.ylabel("absolute error")
plt.title("Small x: naive vs specialized implementation")
plt.legend()
plt.grid(True, which="both", alpha=0.25)
plt.show()

#### Experiment interpretation

**Problem.** A single test value can hide where a numerical method begins to fail.

**Why it happens.** The error of `log(1+x)` grows as `x` becomes small enough that the addition `1+x` loses significant digits.

**Observed behavior.** The log-scale plot compares absolute error against a high-precision reference and shows a clear accuracy advantage for `log1p` in the small-`x` regime.

**Adopted solution.** Evaluate numerical error across the input range that the model will actually encounter, not only at one representative value.

**Takeaway.** *Numerical stability is a property over a range of inputs, not a single example.*


## 10. `log(exp(x))` and Softplus

The slides note that `log(exp(x))` should be simplified to `x`. The reason is not only efficiency: the intermediate `exp(x)` may overflow.

In [ ]:
for value in [1.0, 50.0, 100.0, -100.0]:
    x = torch.tensor(value, dtype=torch.float32)
    naive = torch.log(torch.exp(x))
    print(f"x={value:7.1f}  log(exp(x))={naive.item():>12}  stable={x.item():>8}")

#### Experiment interpretation

**Problem.** The composition `log(exp(x))` is mathematically equal to `x` but can overflow or lose accuracy numerically.

**Why it happens.** The intermediate exponential is materialized before the logarithm can undo it. Large positive values overflow; large negative values can underflow or become inaccurate.

**Observed failure.** At `x=100`, the naive expression becomes `inf`; at `x=-100`, it is already slightly inaccurate.

**Adopted solution.** Simplify the expression algebraically and keep `x` directly. More generally, avoid unstable intermediate representations even when a later operation would theoretically cancel them.

**Takeaway.** *Do not compute an unstable intermediate merely to invert it in the next operation.*


### Softplus

$$
\operatorname{softplus}(x)=\log(1+e^x)
$$

This is a perfect example of a simple formula that requires a stable implementation.

In [ ]:
x = torch.tensor([10.0, 50.0, 100.0], dtype=torch.float32)

naive = torch.log(1 + torch.exp(x))
stable = F.softplus(x)

print("naive :", naive)
print("stable:", stable)

#### Experiment interpretation

**Problem.** The literal Softplus formula `log(1 + exp(x))` overflows for large positive inputs.

**Why it happens.** `exp(100)` overflows in FP32 before the logarithm can bring the value back to a moderate scale.

**Observed failure.** The naive implementation returns `inf` at 100, while `F.softplus` returns the correct asymptotic value near 100.

**Adopted solution.** Use `torch.nn.functional.softplus`, which switches to a numerically stable formulation instead of evaluating the literal expression everywhere.

**Takeaway.** *Built-in nonlinearities often include branch logic specifically to avoid overflow and cancellation.*


# Part II — Stability of probability functions and losses

This is the most important part for practical deep learning: many loss functions combine `exp`, normalization, and `log`.

## 11. LogSumExp

The naive expression

$$
\log \sum_i e^{x_i}
$$

fails if one element is very large (overflow) or if all elements are very negative (underflow).

The stable transformation uses $m=\max_i x_i$:

$$
\log\sum_i e^{x_i} = m + \log\sum_i e^{x_i-m}.
$$

In [ ]:
def naive_logsumexp(x, dim=0):
    return torch.log(torch.exp(x).sum(dim=dim))


def stable_logsumexp_manual(x, dim=0):
    m = x.max(dim=dim, keepdim=True).values
    return (m + torch.log(torch.exp(x - m).sum(dim=dim, keepdim=True))).squeeze(dim)

for x in [
    torch.tensor([1000.0, 1000.0]),
    torch.tensor([-1000.0, -1000.0]),
]:
    print("x =", x.tolist())
    print("naive  =", naive_logsumexp(x))
    print("manual =", stable_logsumexp_manual(x))
    print("torch   =", torch.logsumexp(x, dim=0))
    print()

#### Experiment interpretation

**Problem.** `log(sum(exp(x)))` fails at both extremes: large positive inputs overflow and large negative inputs underflow.

**Why it happens.** The exponentials are formed before the logarithm. The stable identity subtracts `m=max(x)` first, making every exponent non-positive and ensuring that at least one exponent is exactly `exp(0)=1`.

**Observed failure.** The naive result is `inf` for `[1000,1000]` and `-inf` for `[-1000,-1000]`; both the manual stabilized version and `torch.logsumexp` remain finite.

**Adopted solution.** Use `torch.logsumexp` in real code. The manual max-shift implementation is useful to understand why the stabilization works.

**Takeaway.** *The log-sum-exp trick simultaneously controls overflow and catastrophic underflow.*


## 12. Naive vs stable Softmax

A literal implementation of

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

may produce `inf/inf`.

In [ ]:
def softmax_naive(x, dim=-1):
    e = torch.exp(x)
    return e / e.sum(dim=dim, keepdim=True)


def softmax_stable_manual(x, dim=-1):
    z = x - x.max(dim=dim, keepdim=True).values
    e = torch.exp(z)
    return e / e.sum(dim=dim, keepdim=True)

logits = torch.tensor([[1000.0, 999.0, 998.0]])

print("naive :", softmax_naive(logits, dim=1))
print("manual:", softmax_stable_manual(logits, dim=1))
print("F.softmax:", F.softmax(logits, dim=1))

#### Experiment interpretation

**Problem.** A literal Softmax implementation can produce `inf/inf`, which becomes `NaN`.

**Why it happens.** Exponentiating logits near 1000 overflows. Dividing these infinite exponentials by their infinite sum is undefined.

**Observed failure.** The naive softmax is all `NaN`, whereas the max-shifted manual implementation and `F.softmax` return valid probabilities.

**Adopted solution.** Subtract the maximum logit before exponentiating, or simply use `F.softmax`. The shift does not change the mathematical softmax because the same multiplicative factor cancels from numerator and denominator.

**Takeaway.** *Softmax should never be implemented as an unrestricted `exp` followed by normalization.*


## 13. `log(softmax(x))` vs `log_softmax(x)`

This is one of the most important replacements to learn in PyTorch.

In [ ]:
logits = torch.tensor([[1000.0, 0.0, -1000.0]])

bad = torch.log(F.softmax(logits, dim=1))
good = F.log_softmax(logits, dim=1)

print("log(softmax):", bad)
print("log_softmax :", good)

#### Experiment interpretation

**Problem.** `log(softmax(logits))` can lose information after Softmax saturates.

**Why it happens.** With extreme logits, Softmax rounds tiny class probabilities to exactly zero. Taking `log(0)` then produces `-inf`, even though the corresponding log-probability is mathematically finite.

**Observed failure.** The composed operation yields `[0, -inf, -inf]`, while `F.log_softmax` retains the meaningful values `[0, -1000, -2000]`.

**Adopted solution.** Use `F.log_softmax`, which combines the operations using a log-sum-exp formulation and never materializes the tiny probabilities first.

**Takeaway.** *If the next operation is a logarithm, stay in log space.*


The probability of a very unlikely class may be rounded to zero; once that happens, `log(0)` is irreversibly `-inf`. `log_softmax` avoids materializing that extreme probability in the naive way.

## 14. Cross-entropy: use logits, not probabilities

The slides emphasize that cross-entropy should be computed directly from logits.

### Naive implementation

In [ ]:
logits = torch.tensor([[1000.0, 0.0, -1000.0]], requires_grad=True)
target = torch.tensor([1])

p = F.softmax(logits, dim=1)
loss_bad = -torch.log(p[0, target[0]])

print("probabilities =", p)
print("loss_bad      =", loss_bad)

loss_bad.backward()
print("gradient bad  =", logits.grad)

#### Experiment interpretation

**Problem.** Computing cross-entropy from already-normalized probabilities can destroy both the loss and its gradient.

**Why it happens.** The target-class probability underflows to zero after Softmax. `-log(0)` becomes `inf`, and the backward pass encounters undefined combinations that produce `NaN` gradients.

**Observed failure.** The loss is infinite and every logit gradient becomes `NaN`.

**Adopted solution.** Pass raw logits directly to `F.cross_entropy` (or use `F.log_softmax` + `F.nll_loss` when the decomposition is needed). These formulations avoid the unstable probability intermediate.

**Takeaway.** *Loss functions should usually consume logits, not probabilities.*


### Correct implementation with `F.cross_entropy`

In [ ]:
logits = torch.tensor([[1000.0, 0.0, -1000.0]], requires_grad=True)
target = torch.tensor([1])

loss_good = F.cross_entropy(logits, target)
print("loss_good     =", loss_good)

loss_good.backward()
print("gradient good =", logits.grad)

#### Experiment interpretation

**Problem.** We need the same mathematical cross-entropy without saturating the intermediate probabilities.

**Why it happens.** `F.cross_entropy` combines log-softmax and negative log-likelihood in a stable way, so it can represent the very large log-loss directly.

**Observed behavior.** The loss is the finite value `1000`, and the gradient `[1, -1, 0]` remains meaningful.

**Adopted solution.** Use the fused PyTorch loss with logits. Besides being more stable, this is also clearer and usually more efficient than manually composing Softmax and logarithm.

**Takeaway.** *Fused loss functions preserve gradients in regimes where probabilities have already saturated.*


The crucial point is not only to obtain a finite loss: **we need a usable gradient**.

## 15. Binary case: Sigmoid + BCE vs BCEWithLogits

The same principle applies to binary classification.

In [ ]:
z = torch.tensor([100.0], requires_grad=True)
y = torch.tensor([0.0])

p = torch.sigmoid(z)
loss_bad = -(y * torch.log(p) + (1-y) * torch.log(1-p)).mean()

print("sigmoid(z) =", p)
print("manual BCE =", loss_bad)

loss_bad.backward()
print("bad grad   =", z.grad)

#### Experiment interpretation

**Problem.** Manual binary cross-entropy after Sigmoid suffers from the same saturation problem as multiclass Softmax cross-entropy.

**Why it happens.** For `z=100`, `sigmoid(z)` rounds to exactly 1. The term `log(1-p)` therefore evaluates `log(0)`, causing an infinite loss and a `NaN` gradient.

**Observed failure.** The manual BCE is `inf` and the gradient is `NaN`.

**Adopted solution.** Do not apply Sigmoid before the loss. Keep the raw logits and use a numerically stable logits-based BCE formulation.

**Takeaway.** *A saturated probability can erase the information needed by the backward pass.*


In [ ]:
z = torch.tensor([100.0], requires_grad=True)
y = torch.tensor([0.0])

loss_good = F.binary_cross_entropy_with_logits(z, y)
print("BCEWithLogits =", loss_good)

loss_good.backward()
print("good grad     =", z.grad)

#### Experiment interpretation

**Problem.** We want binary cross-entropy to remain finite even for very large positive or negative logits.

**Why it happens.** `binary_cross_entropy_with_logits` combines the Sigmoid and BCE algebraically using a stable softplus/log-sum-exp-style expression.

**Observed behavior.** The loss is finite (`100`) and the gradient is exactly usable (`1`) for this extreme misclassification.

**Adopted solution.** Use `F.binary_cross_entropy_with_logits` or `nn.BCEWithLogitsLoss` and feed it raw logits.

**Takeaway.** *For binary classification, BCE-with-logits is the stable analogue of cross-entropy-with-logits.*


**Practical rule:**

- multiclass → `F.cross_entropy(logits, target)`;
- binary/multilabel → `F.binary_cross_entropy_with_logits(logits, target)` or `nn.BCEWithLogitsLoss`.

## 16. Modern example: attention and extreme softmax values

Scaled dot-product attention contains a softmax:

$$
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V.
$$

With very large scores, a naively implemented softmax can fail.

In [ ]:
Q = torch.tensor([[1000.0, 1000.0], [1000.0, -1000.0]])
K = torch.tensor([[1000.0, 1000.0], [-1000.0, 1000.0]])
V = torch.tensor([[1.0, 0.0], [0.0, 1.0]])

scores = Q @ K.T / math.sqrt(Q.shape[-1])

weights_bad = softmax_naive(scores, dim=-1)
weights_good = F.softmax(scores, dim=-1)

print("scores:\n", scores)
print("\nnaive attention weights:\n", weights_bad)
print("\nstable attention weights:\n", weights_good)
print("\nstable attention output:\n", weights_good @ V)

#### Experiment interpretation

**Problem.** Attention can inherit Softmax overflow when dot-product scores become very large.

**Why it happens.** Large query/key norms produce scores on the order of millions. A naive `exp(scores)` overflows, causing invalid attention weights.

**Observed failure.** The naive attention matrix contains `NaN`, while `F.softmax` produces valid one-hot-like weights.

**Adopted solution.** Use PyTorch's stabilized Softmax or, in production transformer code, optimized scaled-dot-product-attention primitives rather than manually exponentiating scores. Also monitor exploding activations because stabilization prevents `NaN` but does not make extremely large scores desirable.

**Takeaway.** *Stable Softmax prevents arithmetic failure, but extreme logits can still signal a poorly scaled model.*


This directly connects numerical stability to Transformers: a stable softmax is not a low-level implementation detail, but a necessary condition for fundamental blocks of modern architectures to work.

# Part III — Reduced precision and mixed precision

## 17. FP16 vs BF16 vs FP32

FP16 and BF16 both use fewer bits than FP32, but in different ways:

- FP16 has greater mantissa precision than BF16, but a much smaller dynamic range;
- BF16 retains a range similar to FP32, but has coarser precision.

In [ ]:
values = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]

for v in values:
    row = []
    for dtype in [torch.float16, torch.bfloat16, torch.float32]:
        row.append(torch.tensor(v, dtype=dtype).item())
    print(f"{v:>8.1e}   fp16={row[0]:>12.4e}   bf16={row[1]:>12.4e}   fp32={row[2]:>12.4e}")

#### Experiment interpretation

**Problem.** Reduced-precision formats do not lose information in the same way.

**Why it happens.** FP16 has a narrow exponent range, so very small values underflow quickly. BF16 uses fewer fraction bits but an exponent range similar to FP32, so it preserves much smaller magnitudes at the cost of coarser relative precision.

**Observed behavior.** Around `1e-8`, FP16 reaches zero while BF16 still represents a non-zero value.

**Adopted solution.** Choose reduced precision according to the workload. BF16 is often more tolerant of large/small scales; FP16 commonly requires loss/gradient scaling and careful autocasting.

**Takeaway.** *FP16 and BF16 are both 16-bit, but their numerical failure modes are very different.*


### Example of a gradient that underflows in FP16

In [ ]:
g = torch.tensor([1e-8], dtype=torch.float32)
print("float32:", g)
print("float16:", g.to(torch.float16))
print("bfloat16:", g.to(torch.bfloat16))

#### Experiment interpretation

**Problem.** Small gradients may disappear completely in FP16.

**Why it happens.** The value `1e-8` is below the practical representable range used by FP16 here and rounds to zero during conversion.

**Observed failure.** The FP32 and BF16 tensors remain non-zero, but FP16 becomes exactly zero. A zeroed gradient cannot update the parameter.

**Adopted solution.** In mixed-precision training, use gradient/loss scaling so small gradients are multiplied into a representable range before FP16 arithmetic, then unscale before the optimizer update.

**Takeaway.** *Gradient underflow can silently stop learning even when the forward pass looks normal.*


This is the conceptual reason for **gradient scaling**: temporarily multiplying the loss shifts gradients toward magnitudes that are more representable during the backward pass.

## 18. Automatic Mixed Precision (AMP)

On a CUDA GPU, the modern PyTorch pattern is:

1. use `torch.autocast` for the forward pass and loss computation;
2. use `torch.amp.GradScaler` to reduce the risk of FP16 gradient underflow;
3. run backward outside the `autocast` context.

The following cell can be executed in Colab with a GPU runtime; on CPU it is skipped.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    model = nn.Sequential(
        nn.Linear(32, 64),
        nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda")

    x = torch.randn(128, 32, device=device)
    target = torch.randint(0, 10, (128,), device=device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x)
        loss = F.cross_entropy(logits, target)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print("AMP step completed. loss =", loss.item())
else:
    print("CUDA not available: AMP cell skipped. In Colab: Runtime → Change runtime type → GPU.")

#### Experiment interpretation

**Problem.** Pure low-precision training is fast but can overflow activations or underflow gradients.

**Why it happens.** Different operations have different numerical sensitivity. AMP selectively executes suitable operations in reduced precision while retaining higher precision where needed; `GradScaler` increases the scale of the backward signal to protect small FP16 gradients.

**Observed behavior.** On a CUDA runtime the cell demonstrates the standard autocast + scaling training pattern. On CPU-only runtimes it is intentionally skipped.

**Adopted solution.** Use `torch.autocast` and `torch.amp.GradScaler` rather than globally converting every tensor and parameter to FP16.

**Takeaway.** *Mixed precision is a controlled policy, not simply “train everything in float16”.*


## 19. The final result may be representable while intermediate values are not

A common mistake is to think: “if the result lies within the float32 range, then the computation is safe.”

The Euclidean norm contains intermediate squares. For very large components, those squares may overflow even though the final norm itself would be representable.

In [ ]:
a = torch.tensor([1e20, 1e20], dtype=torch.float32)

print("float32 norm:", a.norm())
print("float64 norm:", a.double().norm())

#### Experiment interpretation

**Problem.** A final answer may fit in FP32 even though an intermediate step overflows.

**Why it happens.** A straightforward Euclidean norm can involve squaring `1e20`, producing values around `1e40`, which exceed FP32 range, even though the final norm is only about `1.4e20`.

**Observed failure.** The FP32 norm becomes `inf`, whereas the FP64 computation remains finite.

**Adopted solution.** Use implementations that rescale internally when available, or perform sensitive reductions in a wider dtype. Always reason about intermediate magnitudes, not just the expected final result.

**Takeaway.** *Numerical safety is determined by the whole computation path, including intermediates.*


# Part IV — Linear algebra and conditioning

## 20. Nearly singular matrices

Stability depends not only on the dtype, but also on the **conditioning of the problem**. A nearly singular matrix amplifies small errors in the data and in floating-point rounding.

In [ ]:
A64 = torch.tensor([
    [1.0, 1.0],
    [1.0, 1.0 + 1e-7],
], dtype=torch.float64)
b64 = torch.tensor([2.0, 2.0 + 1e-7], dtype=torch.float64)

A32 = A64.float()
b32 = b64.float()

print("cond(A64) =", torch.linalg.cond(A64).item())
print("solution float64 =", torch.linalg.solve(A64, b64))

try:
    print("solution float32 =", torch.linalg.solve(A32, b32))
except RuntimeError as e:
    print("float32 solve failed:", e)

#### Experiment interpretation

**Problem.** Some problems are intrinsically sensitive to rounding, independent of whether the code is “correct”.

**Why it happens.** The matrix is nearly singular: its condition number is around `4e7`. Small representation errors in FP32 are strongly amplified when solving the system.

**Observed failure.** FP64 returns a solution near `[1,1]`, while FP32 returns `[2,0]` for what appears to be the same system.

**Adopted solution.** Diagnose conditioning with tools such as `torch.linalg.cond`, rescale or reformulate the problem, use regularization when appropriate, and increase precision when the condition number demands it.

**Takeaway.** *More stable arithmetic cannot fully compensate for a badly conditioned mathematical problem.*


### A small perturbation of the data

In [ ]:
A = torch.tensor([
    [1.0, 1.0],
    [1.0, 1.0 + 1e-10],
], dtype=torch.float64)

b1 = torch.tensor([2.0, 2.0 + 1e-10], dtype=torch.float64)
b2 = b1.clone()
b2[1] += 1e-12

x1 = torch.linalg.solve(A, b1)
x2 = torch.linalg.solve(A, b2)

print("x1 =", x1)
print("x2 =", x2)
print("delta solution =", x2 - x1)
print("cond(A) =", torch.linalg.cond(A).item())

#### Experiment interpretation

**Problem.** A tiny perturbation in the input can cause a large perturbation in the solution of an ill-conditioned problem.

**Why it happens.** Near singularity, many very different solutions produce almost the same left-hand side. The inverse mapping therefore amplifies small changes in `b`.

**Observed behavior.** Changing one component of `b` by only `1e-12` shifts the solution by roughly `1e-2`, consistent with the enormous condition number.

**Adopted solution.** Treat condition number as a property of the problem, not just the implementation. Use regularization, better parameterization, scaling, or a numerically appropriate solver.

**Takeaway.** *Sensitivity to input perturbations is distinct from floating-point rounding, although the two effects interact.*


# Part V — Numerical debugging in PyTorch

## 21. Systematically check for `NaN` and `inf`

In [ ]:
def tensor_report(name, x):
    x_detached = x.detach()
    finite = torch.isfinite(x_detached)
    print(f"{name}")
    print("  shape    :", tuple(x_detached.shape))
    print("  dtype    :", x_detached.dtype)
    print("  allfinite:", finite.all().item())
    print("  nan      :", torch.isnan(x_detached).sum().item())
    print("  inf      :", torch.isinf(x_detached).sum().item())
    if finite.any():
        xf = x_detached[finite]
        print("  min/max  :", xf.min().item(), xf.max().item())

x = torch.tensor([1.0, float("inf"), float("nan")])
tensor_report("example", x)

#### Experiment interpretation

**Problem.** `NaN` and `inf` often appear deep inside a training graph, so visual inspection of the final loss is too late.

**Why it happens.** Non-finite values propagate through many tensor operations and can contaminate an entire batch before they become obvious.

**Observed behavior.** The helper reports shape, dtype, counts of `NaN`/`inf`, and finite min/max values, making failures easier to localize.

**Adopted solution.** Add lightweight `torch.isfinite`, `torch.isnan`, and `torch.isinf` checks at strategic boundaries: model inputs, block outputs, loss components, and gradients.

**Takeaway.** *Numerical debugging becomes much easier when invariants are checked close to where values are produced.*


## 22. Check gradients during training

In [ ]:
def report_nonfinite_gradients(model):
    bad = []
    for name, p in model.named_parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            bad.append(name)
    return bad

model = nn.Linear(4, 2)
x = torch.randn(8, 4)
y = torch.randint(0, 2, (8,))

loss = F.cross_entropy(model(x), y)
loss.backward()

print("non-finite gradients:", report_nonfinite_gradients(model))

#### Experiment interpretation

**Problem.** A finite loss does not guarantee that every parameter gradient is finite.

**Why it happens.** Backward computations may contain singularities or unstable branches that are not obvious from the scalar loss alone.

**Observed behavior.** This healthy example returns an empty list, demonstrating the expected baseline behavior of the checker.

**Adopted solution.** Scan `model.named_parameters()` after `backward()` and flag any gradient containing `NaN` or `inf`. In larger models, register hooks or check only selected layers to reduce overhead.

**Takeaway.** *Gradient health should be monitored explicitly when diagnosing unstable training.*


## 23. `detect_anomaly`: locating problematic operations in backward

`torch.autograd.detect_anomaly()` is very useful for debugging, but it is expensive: it should not normally remain enabled during training.

In [ ]:
x = torch.tensor(0.0, requires_grad=True)

try:
    with torch.autograd.detect_anomaly():
        # 0/0 produces NaN in the forward pass and a problematic backward pass
        y = x / x
        y.backward()
except RuntimeError as e:
    print("Anomaly detected:")
    print(str(e).splitlines()[0])

#### Experiment interpretation

**Problem.** Once a backward pass produces `NaN`, it can be difficult to identify which forward operation caused it.

**Why it happens.** Autograd normally propagates gradients without reporting the full origin of every non-finite value.

**Observed behavior.** `detect_anomaly` reports that `DivBackward0` produced `NaN`, tracing the failure back to the deliberate `0/0` operation.

**Adopted solution.** Temporarily wrap suspicious code in `torch.autograd.detect_anomaly()` during debugging. Remove it for normal training because the extra checking has substantial overhead.

**Takeaway.** *Anomaly detection is a localization tool, not a numerical stabilization technique.*


## 24. `nan_to_num` is not a cure

`torch.nan_to_num` can be useful for sanitizing data or outputs in controlled situations, but it **must not hide a numerically unstable loss or gradient**.

In [ ]:
x = torch.tensor([1.0, float("nan"), float("inf"), -float("inf")])
print(torch.nan_to_num(x))

#### Experiment interpretation

**Problem.** Replacing `NaN` and infinities can make a tensor finite without making the computation correct.

**Why it happens.** `nan_to_num` substitutes values after the failure has already occurred. It has no knowledge of the intended mathematics and may replace infinities with extremely large finite numbers.

**Observed behavior.** The tensor becomes finite, but the replacements are arbitrary with respect to the original model semantics.

**Adopted solution.** Use `nan_to_num` only when sanitization is explicitly part of the data/model definition. For training failures, find and stabilize the upstream operation that created the non-finite value.

**Takeaway.** *Finite is not synonymous with correct.*


If a `NaN` appears during training, the priority is to find **the first operation that generates a non-finite value**, rather than replacing it at the end of the pipeline.

# Part VI — Mini-lab: unstable vs stable pipeline

Let us combine the previous ideas in a single multiclass example.

In [ ]:
torch.manual_seed(1)

# Simulate very large logits: a situation that may arise from exploding weights/activations.
logits = 200.0 * torch.randn(16, 5, requires_grad=True)
target = torch.randint(0, 5, (16,))

# Naive pipeline
p = softmax_naive(logits, dim=1)
loss_bad = -torch.log(p[torch.arange(len(target)), target]).mean()

print("bad loss:", loss_bad)
print("bad loss finite?", torch.isfinite(loss_bad).item())

#### Experiment interpretation

**Problem.** Several individually familiar operations can combine into a numerically unstable training pipeline.

**Why it happens.** Very large logits are fed to a naive Softmax, then probabilities are logged. Overflow/saturation creates zeros or `NaN`, and the final averaged loss is non-finite.

**Observed failure.** The naive multiclass pipeline returns a `NaN` loss.

**Adopted solution.** Do not repair the probabilities after Softmax. Eliminate the unstable intermediate and compute the loss directly from logits.

**Takeaway.** *The most effective numerical fix often removes an intermediate representation entirely.*


In [ ]:
# Stable pipeline
logits2 = logits.detach().clone().requires_grad_(True)
loss_good = F.cross_entropy(logits2, target)
loss_good.backward()

print("good loss:", loss_good)
print("good loss finite?", torch.isfinite(loss_good).item())
print("good gradients finite?", torch.isfinite(logits2.grad).all().item())

#### Experiment interpretation

**Problem.** We need the same learning objective while preserving finite values and gradients for extreme logits.

**Why it happens.** `F.cross_entropy` works in a stabilized log-domain formulation instead of explicitly materializing exponentials, normalized probabilities, and logarithms.

**Observed behavior.** The loss remains finite and all gradients pass the `isfinite` check, even though the logits are deliberately very large.

**Adopted solution.** Prefer fused, logits-based PyTorch primitives for common probability/loss computations and validate both the loss and gradients.

**Takeaway.** *Stable primitives are not merely convenience functions; they encode better numerical algorithms.*


# Practical checklist

When you encounter `NaN`, `inf`, a stuck loss, or suspicious gradients:

1. **Check the dtype** (`float16`, `bfloat16`, `float32`, `float64`).
2. Look for `exp`, `log`, `sqrt`, divisions, and subtractions between similar numbers.
3. Inspect intermediate values with `torch.isfinite`.
4. Use loss functions that operate directly on **logits**.
5. Prefer `torch.logsumexp`, `F.log_softmax`, `F.cross_entropy`, and `BCEWithLogitsLoss` over manual compositions.
6. Check whether normalization uses an appropriate `eps`.
7. With FP16, consider underflow/overflow and use AMP + gradient scaling.
8. For linear algebra problems, inspect the **condition number**.
9. Use `detect_anomaly()` temporarily to locate a problematic backward pass.
10. Do not use `nan_to_num` to mask a numerically unstable pipeline.

# Exercises

### Exercise 1 — overflow threshold
Write a function that, given a `torch.dtype`, estimates the largest `x` for which `exp(x)` remains finite. Verify the result experimentally.

### Exercise 2 — summation
Build a vector containing both very large and very small numbers. Compare `sum` in float32 and float64, and try different orderings of the elements.

### Exercise 3 — variance
Generate data of the form `offset + noise`, progressively increasing `offset`. Compare:

```python
(x*x).mean() - x.mean()**2
```

with `torch.var`.

### Exercise 4 — cross-entropy
Progressively increase the scale of the logits from 1 to 1000 and determine when the `softmax -> log` pipeline starts producing non-finite values.

### Exercise 5 — mixed precision
On a GPU, compare a small training loop in FP32 and with AMP. Record the loss, runtime, and whether non-finite gradients occur.

### Exercise 6 — attention
Increase the norm of `Q` and `K` and compare the naive softmax with `F.softmax`.

# References

- Course slides: *04_numerical.pptx*, section **Numerical Precision: A deep learning super skill** and the following sections.
- Starting notebook indicated by the instructor: `serivan/DeepLearning/02-Preliminaries/numerical_errors.ipynb`.
- PyTorch documentation: *Numerical accuracy*, `torch.finfo`, `torch.logsumexp`, `F.log_softmax`, `BCEWithLogitsLoss`, *Automatic Mixed Precision*.

> Note: this version is a complete PyTorch reconstruction aligned with the contents of the slides. It is not a mechanical cell-by-cell conversion of the remote notebook.